# 7.2 Capstone Part 2: Build an Agentic Chatbot with Tools

Interactive notebook for the **How AI Works** course. Run each cell top to bottom (Shift+Enter). The text and explanations live in the lesson page.

In [ ]:
# Setup: install the one dependency this course uses.
%pip install numpy --quiet

In [ ]:
import numpy as np
import datetime
import math

# --- 1. Knowledge Base & Vector Search (from 7.1) ---
knowledge_base = [
    "The dot product measures how aligned two vectors are by multiplying corresponding elements and summing.",
    "Embeddings turn words and sentences into dense vectors where semantic similarity corresponds to geometric distance.",
    "Attention lets a transformer weigh how much each token should look at every other token in the prompt.",
    "Softmax turns a list of real-valued logits into a probability distribution that sums to exactly 1.0.",
    "Retrieval-Augmented Generation (RAG) fetches relevant external documents and injects them into the prompt.",
    "Model Context Protocol (MCP) is an open client-server standard connecting AI models to external tools and data.",
]

def simple_embed(text):
    """Toy bag-of-words character frequency embedding normalized to unit length."""
    chars = "abcdefghijklmnopqrstuvwxyz"
    counts = np.array([text.lower().count(c) for c in chars], dtype=float)
    norm = np.linalg.norm(counts)
    return counts / (norm + 1e-9)

doc_vectors = np.array([simple_embed(doc) for doc in knowledge_base])

def tool_rag_search(query: str) -> str:
    """Search the course knowledge base for relevant facts."""
    q_vec = simple_embed(query)
    scores = np.dot(doc_vectors, q_vec)
    best_idx = int(np.argmax(scores))
    if scores[best_idx] < 0.2:
        return "No confident matching information found in knowledge base."
    return knowledge_base[best_idx]

# --- 2. Calculator Tool ---
def tool_calculator(expression: str) -> str:
    """Evaluate safe math calculations."""
    try:
        # Whitelisted safe math namespace
        allowed = {"abs": abs, "round": round, "math": math, "np": np}
        clean_expr = expression.replace("^", "**")
        result = eval(clean_expr, {"__builtins__": {}}, allowed)
        return f"Result = {result}"
    except Exception as e:
        return f"Math error: {str(e)}"

# --- 3. System Time Tool ---
def tool_get_time(query: str = "") -> str:
    """Get the current date and time."""
    now = datetime.datetime.now()
    return f"Current date and time: {now.strftime('%Y-%m-%d %H:%M:%S')}"

# Register available tools
TOOL_REGISTRY = {
    "rag_search": {
        "fn": tool_rag_search,
        "description": "Look up facts about AI, vectors, embeddings, attention, softmax, and RAG."
    },
    "calculator": {
        "fn": tool_calculator,
        "description": "Evaluate math calculations like 45 * 32, sqrt(144), or 2**10."
    },
    "get_time": {
        "fn": tool_get_time,
        "description": "Check current date or time."
    }
}

print(f"Registered {len(TOOL_REGISTRY)} tools: {list(TOOL_REGISTRY.keys())}")

In [ ]:
import re

def agent_reason(query: str, history: list[str]) -> tuple[str, str, str]:
    """
    Simulates the model's internal reasoning.
    Returns: (thought, action, action_input)
    If no tool is needed, action is 'FINAL_ANSWER'.
    """
    history_str = " ".join(history)
    combined = (query + " " + history_str).lower()

    # Did the user ask for math and haven't calculated yet?
    math_match = re.search(r"(\d+[\s\+\-\*\/\^\%]+\d+)", query)
    if math_match and "Result =" not in history_str:
        expr = math_match.group(1).strip()
        thought = f"The user query contains a math expression '{expr}'. I need to evaluate it."
        return thought, "calculator", expr

    # Did the user ask about AI concepts and haven't retrieved yet?
    ai_keywords = ["vector", "dot product", "embedding", "attention", "softmax", "rag", "mcp"]
    if any(k in query.lower() for k in ai_keywords) and not any("knowledge base" in h for h in history):
        thought = "The user is asking about an AI concept. I should query the knowledge base."
        return thought, "rag_search", query

    # Did the user ask for the time?
    if any(k in query.lower() for k in ["time", "date", "today"]) and "Current date" not in history_str:
        thought = "The user is asking for temporal information. I will check the system clock."
        return thought, "get_time", ""

    # If all needed info is collected, formulate the final answer
    thought = "I have collected all necessary facts and observations to answer the query."
    return thought, "FINAL_ANSWER", ""

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    print(f"=== User: '{user_query}' ===")
    history = []
    
    for step in range(1, max_iterations + 1):
        thought, action, action_input = agent_reason(user_query, history)
        print(f"\n[Step {step}]")
        print(f"  💭 Thought: {thought}")
        
        if action == "FINAL_ANSWER":
            print("  🎯 Decision: Formulate Final Answer")
            break
            
        print(f"  ⚡ Action: {action}('{action_input}')")
        
        # Execute tool
        if action in TOOL_REGISTRY:
            tool_fn = TOOL_REGISTRY[action]["fn"]
            observation = tool_fn(action_input)
        else:
            observation = f"Error: Tool '{action}' not recognized."
            
        print(f"  👁 Observation: {observation}")
        history.append(f"Observation from {action}: {observation}")
    
    # Synthesize observations into a final response
    if history:
        summary = "\n".join([f"- {h}" for h in history])
        final_answer = f"Based on my tools and observations:\n{summary}"
    else:
        final_answer = f"I answered directly without requiring external tools."
        
    print(f"\n[Final Response]\n{final_answer}\n")
    return final_answer

In [ ]:
run_agent("What is the dot product, and what is 24 * 15?")

## Exercises

**Exercise 1: Add a Tool.** Add a new tool named `word_count` to `TOOL_REGISTRY` that counts how many words are in a string. Test the tool directly.

<details>
<summary>Show solution</summary>

```python
def tool_word_count(text: str) -> str:
    words = text.strip().split()
    return f"Word count = {len(words)}"

TOOL_REGISTRY["word_count"] = {
    "fn": tool_word_count,
    "description": "Counts the number of words in a text snippet."
}

print(tool_word_count("Artificial Intelligence from math to multi-agent systems"))
```

Expected output:
```text
Word count = 7
```

</details>

**Exercise 2: Prevent Infinite Loops.** Why is `max_iterations` essential when deploying autonomous agents in the real world?

<details>
<summary>Show solution</summary>

If an agent encounters an unresolvable error (such as a failing tool, contradictory instructions, or unexpected observations), it might continue reasoning and invoking tools indefinitely. 

Setting `max_iterations` guarantees that execution halts safely, preventing runaway compute costs, API quota exhaustion, and unresponsive processes.

</details>

---

## Course Complete!

You have traversed the entire landscape of modern Artificial Intelligence:

1. **Foundations:** Vectors, dot products, matrices, and neurons.
2. **Representations:** Tokens, embeddings, and geometric vector spaces.
3. **The Core Engine:** Transformers, attention, next-token prediction, loss, and gradients.
4. **Context & Retrieval:** Prompt structure, context windows, and RAG.
5. **Autonomy:** Tools, Model Context Protocol (MCP), reasoning loops, and multi-agent coordination.

You now possess the foundational knowledge behind every major advancement in modern language models and autonomous agentic systems.

*Try each exercise in a new cell below before opening the solution.*